[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week04.ipynb)

# Week 4: Data Pipelines
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

The Dataset and DataLoader abstractions; batching, shuffling, transforms, and splits.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Build a custom Dataset and DataLoader.
- Reason about batching, shuffling, and clean splits.
- Recognize and avoid data leakage.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Write a custom Dataset and DataLoader live and iterate over batches.

In [ ]:
# A custom Dataset + DataLoader, then iterate batches
from torch.utils.data import Dataset, DataLoader

class ToyData(Dataset):
    def __init__(self, n=100):
        self.x = torch.randn(n, 3); self.y = torch.randint(0, 2, (n,))
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

ds = ToyData()
dl = DataLoader(ds, batch_size=16, shuffle=True)
xb, yb = next(iter(dl))
print("dataset size:", len(ds), "| one batch:", tuple(xb.shape), tuple(yb.shape))

## Demonstration 2
Show how batch size and shuffling change each epoch.

In [ ]:
# Batch size sets steps per epoch; shuffling reorders each epoch
for bs in [8, 32]:
    print(f"batch_size={bs}: {len(DataLoader(ds, batch_size=bs))} batches per epoch")
dl = DataLoader(ds, batch_size=4, shuffle=True)
print("epoch 1 first labels:", next(iter(dl))[1].tolist())
print("epoch 2 first labels:", next(iter(dl))[1].tolist())

## Demonstration 3
Introduce a data leak (normalizing on the full dataset), show the inflated metric, then fix it.

In [ ]:
# Data leak: normalizing with whole-dataset stats vs train-only stats
data = torch.randn(100, 1) * 5 + 10
train, test = data[:80], data[80:]

mu, sd = data.mean(), data.std()                 # LEAK: uses test data
print("leaky test mean ~ 0:", round(((test - mu) / sd).mean().item(), 3))

mu, sd = train.mean(), train.std()               # correct: train only
print("correct test mean (not 0):", round(((test - mu) / sd).mean().item(), 3))

---
Student materials for this week: the lab handout (`labs/week04.html`) and the curated references (`references/week04.html`) in the course site.